In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

print("Starting Bronze Layer Ingestion...")
print("="*60)

# Read the parquet file you uploaded
df = spark.read.parquet("abfss://8b73c65d-76d6-466c-96b4-ed517828198f@onelake.dfs.fabric.microsoft.com/bffa460f-5aa4-4fec-ad78-8bb5cf9b4505/Files/auckland_crashes_api_filtered.parquet")

print(f"Loaded {df.count():,} records from parquet file")
print(f"Columns: {len(df.columns)}")

# Show sample
print("\nSample data:")
df.show(5, truncate=False)

StatementMeta(, 00fcaf2c-542b-4fde-afe7-de044106621a, 3, Finished, Available, Finished, False)

Starting Bronze Layer Ingestion...
Loaded 120,579 records from parquet file
Columns: 72

Sample data:
+--------+-------------+----------+-------+------+---+---------------+---------+-------------------------+------------------+---------------+--------------------+-----------------+----------------+------------------+---------+------+------------------------+-----+----------+-----+---------+---------+-------+---------------+------------+----+----------+-----------+----------------+-----+----------+-------------+---------------------+-----------+----------------+--------+-------------+----------+-----------+----------+---------------+-------------+--------+-----------+---------+---------+------------------+-----------+----------+-----------+-----------+---+----+-------------------+-----+--------+--------------+-------------+-----------+-----+----+-----+------------------+-----+------------+-------+----------+----------+--------+------------------+-------------------+
|OBJECTID|advisorySp

In [2]:
# Add metadata for data lineage tracking
df_bronze = df \
    .withColumn("ingestion_timestamp", F.current_timestamp()) \
    .withColumn("source_file", F.lit("auckland_crashes_api_filtered.parquet")) \
    .withColumn("data_source", F.lit("NZTA CAS Open Data Portal")) \
    .withColumn("year_range", F.lit("2015-2025")) \
    .withColumn("bronze_record_id", F.monotonically_increasing_id())

print(f"Added metadata columns")
print(f"Total columns now: {len(df_bronze.columns)}")

StatementMeta(, 00fcaf2c-542b-4fde-afe7-de044106621a, 4, Finished, Available, Finished, False)

Added metadata columns
Total columns now: 77


In [3]:
# Write as Delta table (Bronze layer)
print("\nWriting to Delta table...")

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze_crashes_raw")

print("Bronze table created: bronze_crashes_raw")

# Verify
bronze_count = spark.table("bronze_crashes_raw").count()
print(f"Verified: {bronze_count:,} records in bronze table")

StatementMeta(, 00fcaf2c-542b-4fde-afe7-de044106621a, 5, Finished, Available, Finished, False)


Writing to Delta table...
Bronze table created: bronze_crashes_raw
Verified: 120,579 records in bronze table


In [4]:
# Check data quality
bronze_df = spark.table("bronze_crashes_raw")

print("\nBRONZE LAYER DATA QUALITY REPORT")
print("="*60)

# 1. Record count
print(f"Total records: {bronze_df.count():,}")

# 2. Date range
print("\nDate range:")
bronze_df.select(
    F.min("crashYear").alias("earliest_year"),
    F.max("crashYear").alias("latest_year")
).show()

# 3. Missing coordinates
null_coords = bronze_df.filter(
    F.col("X").isNull() | F.col("Y").isNull()
).count()
print(f"\nRecords with missing coordinates: {null_coords:,}")

# 4. Severity distribution
print("\nSeverity distribution:")
bronze_df.groupBy("crashSeverity") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

# 5. Geographic bounds check
print("\nGeographic bounds:")
bronze_df.select(
    F.min("X").alias("min_longitude"),
    F.max("X").alias("max_longitude"),
    F.min("Y").alias("min_latitude"),
    F.max("Y").alias("max_latitude")
).show()

print("\nBronze layer quality check complete!")

StatementMeta(, 00fcaf2c-542b-4fde-afe7-de044106621a, 6, Finished, Available, Finished, False)


BRONZE LAYER DATA QUALITY REPORT
Total records: 120,579

Date range:
+-------------+-----------+
|earliest_year|latest_year|
+-------------+-----------+
|         2015|       2025|
+-------------+-----------+


Records with missing coordinates: 0

Severity distribution:
+----------------+-----+
|   crashSeverity|count|
+----------------+-----+
|Non-Injury Crash|85718|
|     Minor Crash|28627|
|   Serious Crash| 5745|
|     Fatal Crash|  489|
+----------------+-----+


Geographic bounds:
+------------------+----------------+------------------+------------------+
|     min_longitude|   max_longitude|      min_latitude|      max_latitude|
+------------------+----------------+------------------+------------------+
|174.18858337404384|175.511927310006|-37.28577804475671|-36.12149106374211|
+------------------+----------------+------------------+------------------+


Bronze layer quality check complete!
